# Phase 2 Step 1: Verification of `clean.py`

This notebook verifies the mathematical and physical operations implemented in `backend/app/clean.py`:
1. **Zodiacal Background Subtraction:** $I_{\text{clean}} = I_{\text{IMAGE}} - I_{\text{ZODI}}$
2. **Uncertainty Propagation:** $\sigma = \sqrt{\text{VARIANCE}}$
3. **Bitmask Quality Filtering:** Defect masking with strict Bit 21 (`SOURCE`) preservation
4. **Spectral Coordinate Solution:** WCS key `"W"` with SIP disabled to calculate $(\lambda, \Delta\lambda)$

In [1]:
from pathlib import Path
import numpy as np
from astropy.io import fits
from backend.app.clean import clean_cutout, wavelength_at
from backend.app.flags import bad_pixel_bitmask

print("clean.py and dependencies successfully imported.")

clean.py and dependencies successfully imported.


In [2]:
# Locate sample Barnard's Star D2 cutout from the research cache
sample_path = Path("/Users/govindapandey/Desktop/Desktop/SphereX_PlanetX/cache/cutouts/2025W35_1A_0444_1_D2_269.44630_+4.76820_0.0800.fits")
assert sample_path.exists(), f"Sample cutout not found at {sample_path}"
print(f"Using verified Level-2 SPHEREx cutout: {sample_path.name}")

Using verified Level-2 SPHEREx cutout: 2025W35_1A_0444_1_D2_269.44630_+4.76820_0.0800.fits


In [3]:
# Execute clean_cutout on Barnard's Star coordinates (RA=269.4463, Dec=4.7682)
ra_target, dec_target = 269.4463, 4.7682
result = clean_cutout(sample_path, ra=ra_target, dec=dec_target)

print("clean_cutout executed successfully. Returned keys:")
for k, v in result.items():
    if isinstance(v, np.ndarray):
        print(f" - {k}: shape {v.shape}, dtype {v.dtype}")
    else:
        print(f" - {k}: {v}")

clean_cutout executed successfully. Returned keys:
 - image: shape (47, 47), dtype float64
 - sigma: shape (47, 47), dtype float64
 - bad: shape (47, 47), dtype bool
 - wcs: WCS Keywords

Number of WCS axes: 2
CTYPE : 'RA---TAN-SIP' 'DEC--TAN-SIP'
CUNIT : 'deg' 'deg'
CRVAL : 269.235288438 5.41782201993
CRPIX : -106.5 -354.5
PC1_1 PC1_2  : 0.00170304081044 -3.37341041361e-05
PC2_1 PC2_2  : -3.46578978814e-05 -0.00170573693728
CDELT : 1.0 1.0
NAXIS : 47  47
 - mjd: 60914.276937069
 - wl_um: 1.2384
 - bw_um: 0.0298


In [4]:
# Rigorous Mathematical & Physical Verifications
with fits.open(sample_path) as hdul:
    raw_image = hdul["IMAGE"].data.astype(float)
    zodi = hdul["ZODI"].data.astype(float)
    variance = hdul["VARIANCE"].data.astype(float)
    flags = hdul["FLAGS"].data
    header = hdul["IMAGE"].header
    mask = bad_pixel_bitmask(hdul["FLAGS"].header)

# 1. Verify Zodiacal Subtraction
expected_image = raw_image - zodi
np.testing.assert_allclose(result["image"], expected_image, err_msg="Zodiacal subtraction does not match IMAGE - ZODI!")

# 2. Verify Uncertainty Propagation
expected_sigma = np.where(variance > 0, np.sqrt(variance), np.nan)
np.testing.assert_allclose(result["sigma"], expected_sigma, err_msg="Noise sigma does not match sqrt(VARIANCE)!")

# 3. Verify Quality Bitmask Masking
expected_bad = (flags & mask) != 0
assert np.array_equal(result["bad"], expected_bad), "Bad pixel mask does not match bitmask logic!"
assert (mask & (1 << 21)) == 0, "Bit 21 (SOURCE) must never be in bad pixel mask!"

# 4. Verify Wavelength Solution for D2 (Band 2: 1.10 - 1.62 µm)
assert result["wl_um"] is not None and 1.10 <= result["wl_um"] <= 1.62, f"Wavelength {result['wl_um']} out of band 2 range!"
assert result["bw_um"] is not None and result["bw_um"] > 0, f"Invalid bandwidth {result['bw_um']}!"

# 5. Verify Timing
assert result["mjd"] == float(header["MJD-AVG"]), "Timestamp must be MJD-AVG!"

print("All mathematical, astrophysical, and bitmask checks PASSED successfully!.")

All mathematical, astrophysical, and bitmask checks PASSED successfully!.
